In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score


In [2]:
df = pd.read_csv("/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/03_walmart/WALMART_SALES/walmart_sales.csv")

In [3]:
df = df.dropna(subset=["Date"])
df = df.dropna(subset=["Weekly_Sales"])

In [4]:
df["Store"] = df["Store"].astype(int)
store_counts = df["Store"].value_counts()
print(store_counts)

Store
3     10
14     9
13     9
19     8
1      8
18     7
7      7
5      7
6      6
4      6
8      6
2      6
12     5
17     5
20     4
16     4
9      4
15     3
10     3
11     1
Name: count, dtype: int64


In [5]:
mean = df['Unemployment'].mean()
sigma = df['Unemployment'].std()
mask = df['Unemployment'].between(mean - 3 * sigma, mean + 3 * sigma)
df = df[mask]

In [6]:
df = df[df["Store"].isin(store_counts[store_counts >= 2].index)]
print(df["Store"].value_counts())

Store
3     10
13     9
19     8
14     7
7      7
1      7
6      6
4      6
18     6
5      6
8      5
2      5
20     4
16     4
10     3
17     3
9      3
15     2
Name: count, dtype: int64


In [7]:
df['date'] = pd.to_datetime(df['Date'], format = "%d-%m-%Y")
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['weekday'] = df['date'].dt.dayofweek
df = df.drop(columns=['Date'])

In [8]:
df["year"] = df["year"].astype(int)
df["month"] = df["month"].astype(int)
df["day"] = df["day"].astype(int)
df["weekday"] = df["weekday"].astype(int)

In [9]:
df["Holiday_Flag"] = df["Holiday_Flag"].astype("Int64")

### Split 

In [10]:
X = df.drop(columns=["Weekly_Sales"])
y = df["Weekly_Sales"]

In [11]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=X["Store"])

In [12]:
numeric_features = ["Temperature", "Fuel_Price", "CPI", "Unemployment", "year", "month", "day", "weekday"]
categorical_features = ["Store", "Holiday_Flag"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()) 
])

# Catégories globales connues avant la validation croisée afin d'éviter qu'un magasin absent d'un pli d'entraînement soit considéré 
# comme une catégorie inconnue pendant la validation.
store_categories = np.sort(X["Store"].dropna().unique())
holiday_categories = np.sort(X["Holiday_Flag"].dropna().unique())

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        categories=[store_categories, holiday_categories],
        drop="first",
        handle_unknown="ignore",
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [13]:
cv = KFold( 
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [14]:
# Paramètres pour Linear Regression
params_lr = {
    "model__fit_intercept": [True, False],
    "model__tol": [1e-4, 1e-3, 1e-2],
}

# Paramètres pour Ridge
params_ridge = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__solver": ["auto", "svd", "cholesky", "lsqr"],
    "model__fit_intercept": [True, False],
    "model__max_iter": [100000, 500000],
    "model__tol": [1e-4, 1e-3, 1e-2],
}

# Paramètres pour Lasso
params_lasso = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__max_iter": [100000, 500000],
    "model__tol": [1e-4, 1e-3, 1e-2],
}

# Paramètres pour ElasticNet
params_elasticnet = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0],
    "model__l1_ratio": [0.1, 0.5, 0.9],
    "model__max_iter": [100000, 500000],
    "model__tol": [1e-4, 1e-3, 1e-2],
}

In [15]:
pipeline_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

pipeline_ridge = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge()),
])

pipeline_lasso = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso()),
])

pipeline_elasticnet = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ElasticNet()),
])

In [16]:
models = {
    "Linear Regression": (pipeline_lr, params_lr),
    "Ridge": (pipeline_ridge, params_ridge),
    "Lasso": (pipeline_lasso, params_lasso),
    "ElasticNet": (pipeline_elasticnet, params_elasticnet)
}

In [17]:
scoring = {
    "R2": "r2",
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
}

In [18]:
grid_results = {}

for model_name, (pipeline, params) in models.items():

    print(f"Performing Grid Search for {model_name} : ")
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring=scoring,
        refit="R2",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(x_train, y_train)
    grid_results[model_name] = grid_search
    
    print(f"Best parameters for {model_name}: {grid_search.best_params_}")
    print(f"Best R2 score for {model_name}: {grid_search.best_score_:.4f}")
    print()

Performing Grid Search for Linear Regression : 
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best parameters for Linear Regression: {'model__fit_intercept': True, 'model__tol': 0.0001}
Best R2 score for Linear Regression: 0.7376

Performing Grid Search for Ridge : 
Fitting 5 folds for each of 240 candidates, totalling 1200 fits
Best parameters for Ridge: {'model__alpha': 0.1, 'model__fit_intercept': True, 'model__max_iter': 100000, 'model__solver': 'lsqr', 'model__tol': 0.01}
Best R2 score for Ridge: 0.7946

Performing Grid Search for Lasso : 
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters for Lasso: {'model__alpha': 100.0, 'model__max_iter': 100000, 'model__tol': 0.01}
Best R2 score for Lasso: 0.7514

Performing Grid Search for ElasticNet : 
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best parameters for ElasticNet: {'model__alpha': 0.01, 'model__l1_ratio': 0.9, 'model__max_iter': 100000, 'model__tol': 0.0001}
Best R2 sco

In [19]:
def summarize_grid(grid, model_name, metrics):

    results = pd.DataFrame(grid.cv_results_)
    best = results.iloc[grid.best_index_]

    summary = {
        "Model": model_name,
        "Best Params": grid.best_params_
    }

    print(f"\n===== {model_name} =====")
    print(f"Meilleurs paramètres : {grid.best_params_}")

    for metric, sign in metrics.items():

        mean = best[f"mean_test_{metric}"] * sign
        std = best[f"std_test_{metric}"]

        summary[metric] = mean
        summary[f"{metric} std"] = std

        print(f"{metric:<8}: {mean:.4f} (+/- {std:.4f})")

    return (
        summary,
        results.sort_values(f"rank_test_{grid.refit}")
    )

In [20]:
metrics = {
    "R2": 1,
    "MAE": -1,
    "RMSE": -1
}

summary_rows = []
detailed_results = {}

for model_name, grid in grid_results.items():

    summary, details = summarize_grid(
        grid,
        model_name,
        metrics
    )

    summary_rows.append(summary)
    detailed_results[model_name] = details

df_grid = (
    pd.DataFrame(summary_rows)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)
pd.set_option("display.max_colwidth", None)
display(df_grid.round(4))


===== Linear Regression =====
Meilleurs paramètres : {'model__fit_intercept': True, 'model__tol': 0.0001}
R2      : 0.7376 (+/- 0.2463)
MAE     : 187470.3287 (+/- 84572.6644)
RMSE    : 299617.6185 (+/- 165262.9913)

===== Ridge =====
Meilleurs paramètres : {'model__alpha': 0.1, 'model__fit_intercept': True, 'model__max_iter': 100000, 'model__solver': 'lsqr', 'model__tol': 0.01}
R2      : 0.7946 (+/- 0.1520)
MAE     : 185077.2020 (+/- 56841.7946)
RMSE    : 279887.8302 (+/- 110019.7561)

===== Lasso =====
Meilleurs paramètres : {'model__alpha': 100.0, 'model__max_iter': 100000, 'model__tol': 0.01}
R2      : 0.7514 (+/- 0.2401)
MAE     : 183597.8585 (+/- 85744.6925)
RMSE    : 288790.3143 (+/- 164852.3550)

===== ElasticNet =====
Meilleurs paramètres : {'model__alpha': 0.01, 'model__l1_ratio': 0.9, 'model__max_iter': 100000, 'model__tol': 0.0001}
R2      : 0.7824 (+/- 0.1866)
MAE     : 180593.8210 (+/- 66364.7016)
RMSE    : 280941.3629 (+/- 129826.7326)


,Model,Best Params,R2,R2 std,MAE,MAE std,RMSE,RMSE std
0,Ridge,"{'model__alpha': 0.1, 'model__fit_intercept': True, 'model__max_iter': 100000, 'model__solver': 'lsqr', 'model__tol': 0.01}",0.7946,0.1520,185077.2020,56841.7946,279887.8302,110019.7561
1,ElasticNet,"{'model__alpha': 0.01, 'model__l1_ratio': 0.9, 'model__max_iter': 100000, 'model__tol': 0.0001}",0.7824,0.1866,180593.8210,66364.7016,280941.3629,129826.7326
2,Lasso,"{'model__alpha': 100.0, 'model__max_iter': 100000, 'model__tol': 0.01}",0.7514,0.2401,183597.8585,85744.6925,288790.3143,164852.3550
3,Linear Regression,"{'model__fit_intercept': True, 'model__tol': 0.0001}",0.7376,0.2463,187470.3287,84572.6644,299617.6185,165262.9913


## Test du modèle sur le jeu de test :

In [21]:
print(grid_results["Ridge"].best_params_)
display(grid_results["Ridge"].best_estimator_)

{'model__alpha': 0.1, 'model__fit_intercept': True, 'model__max_iter': 100000, 'model__solver': 'lsqr', 'model__tol': 0.01}


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['Store','Holiday_Flag','Temperature',...,'month','day','weekday']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis co

In [22]:
best_estimator = grid_results["Ridge"].best_estimator_
best_params = grid_results["Ridge"].best_params_

best_model_params = {
    key: best_estimator.get_params()[key]
    for key in best_params
}

print(best_params)
print(best_model_params)

{'model__alpha': 0.1, 'model__fit_intercept': True, 'model__max_iter': 100000, 'model__solver': 'lsqr', 'model__tol': 0.01}
{'model__alpha': 0.1, 'model__fit_intercept': True, 'model__max_iter': 100000, 'model__solver': 'lsqr', 'model__tol': 0.01}


In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Meilleur modèle Ridge sélectionné par le Grid Search
best_ridge = grid_results["Ridge"].best_estimator_

# Prédictions sur les jeux d'entraînement et de test
y_train_pred = best_ridge.predict(x_train)
y_test_pred = best_ridge.predict(x_test)

# Métriques sur le train
r2_train = r2_score(y_train, y_train_pred)
mae_train = mean_absolute_error(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))

# Évaluation finale
r2_test = r2_score(y_test, y_test_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

print("Résultats sur le jeu d'entraînement :")
print(f"R²   : {r2_train:.4f}")
print(f"MAE  : {mae_train:.2f}")
print(f"RMSE : {rmse_train:.2f}")

print("\nRésultats sur le jeu de test :")
print(f"R²   : {r2_test:.4f}")
print(f"MAE  : {mae_test:.2f}")
print(f"RMSE : {rmse_test:.2f}")

print("\nÉcart train - test :")
print(f"Écart de R²   : {(r2_train - r2_test) * 100:.4f}%")
print(f"Écart de MAE  : {mae_test - mae_train:.2f}")
print(f"Écart de RMSE : {rmse_test - rmse_train:.2f}")

Résultats sur le jeu d'entraînement :
R²   : 0.9712
MAE  : 85376.66
RMSE : 116475.35

Résultats sur le jeu de test :
R²   : 0.9515
MAE  : 99587.56
RMSE : 133578.69

Écart train - test :
Écart de R²   : 1.9704%
Écart de MAE  : 14210.90
Écart de RMSE : 17103.34


## ==> La différence de score entre le train et le test est de 1.97%. Ce qui est un bon indicateur que le modèle est bien régularisé et qu'il n'y a pas de sur-apprentissage (overfitting).